# Zero-Shot Image Segmentation using Segment Anything Model (SAM)

Companion notebook for the To Data & Beyond tutorial.


## Setup notes

- Designed for a hosted notebook runtime with internet access; GPU acceleration is recommended.
- The notebook downloads the public tutorial image and the SlimSAM checkpoint from Hugging Face.
- Package APIs and model availability can change after publication.
- No credentials or saved execution outputs are included.
- The notebook was checked structurally; it was not execution-tested end to end in this repository.


Zero-shot image segmentation—the ability to segment objects in an image without prior training on those specific objects—has become an exciting new frontier in computer vision. The Segment Anything Model (SAM) from Meta AI has emerged as a powerful tool for tackling this challenge.


This article provides a comprehensive guide on leveraging SAM for zero-shot image segmentation. We’ll start by introducing the fundamentals of image segmentation and why zero-shot capabilities are so valuable. Then, we’ll walk through setting up the necessary working environment and dependencies to use SAM.


The core of the article focuses on generating segmentation masks using SAM, exploring both full-image and single-point inference modes. We’ll discuss techniques for faster inference and optimizing performance, enabling you to efficiently apply SAM in your own projects.


This tutorial will benefit a wide audience, from machine learning practitioners exploring the latest advancements in computer vision to developers looking to incorporate state-of-the-art segmentation into their applications. By the end, you’ll have the knowledge and hands-on experience to harness the power of SAM for your zero-shot image segmentation needs.


## Introduction to Image Segmentation


*Image segmentation divides a scene into pixel-level regions, making its objects and boundaries easier to analyze.*


Image segmentation is a process in digital image processing and computer vision that involves dividing an image into multiple segments, regions, or objects. It is used to simplify and change the representation of an image to make it easier to analyze and extract features from.


Image segmentation is often used to locate objects and boundaries in images. It can be applied to a single image or a stack of images, as is common in medical imaging. The output of image segmentation is a set of segments that collectively cover the entire image or a set of contours that can be used to create 3D reconstructions.


Image segmentation has a wide range of applications, including:


- Medical imaging (e.g. cancer cell segmentation)
- Self-driving systems (e.g. lane segmentation and pedestrian identification)
- Satellite imaging and remote sensing
- Content-based image retrieval
- Fingerprint recognition
- Video object co-segmentation and action localization


There are two main classes of segmentation techniques: classical computer vision approaches and groups of image segmentation. The latter includes:


- Semantic segmentation: assigning each pixel a class label
- Instance segmentation: identifying the specific instance of the object that each pixel belongs to
- Panoptic segmentation: a combination of semantic and instance segmentation


*Semantic segmentation labels categories, instance segmentation separates individual objects, and panoptic segmentation combines both views.*


Image segmentation can be performed using various techniques, including:


- Edge-based segmentation: identifying the edges of objects in an image
- Threshold-based segmentation: dividing pixels based on their intensity relative to a given threshold
- Region-based segmentation: dividing an image into regions with similar characteristics
- Cluster-based segmentation: using clustering algorithms to identify hidden information in images
- Watershed segmentation: treating images like topographic maps and dividing them into regions based on pixel brightness


## Setting up the Working Environment


Let’s start by setting up the working environment. First, we will install the Transformers package, PyTorch dependencies, and Gradio for demo deployment of the application.


In [ ]:
!pip install transformers
!pip install gradio
!pip install timm
!pip install torchvision


Next, we will define helper functions that we will use throughout this article to plot points, boxes, and masks on the segmented parts of the images.


In [ ]:
import io
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image


def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(
        plt.Rectangle(
            (x0, y0), w, h,
            edgecolor="green", facecolor=(0, 0, 0, 0), lw=2
        )
    )


def show_boxes_on_image(raw_image, boxes):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    for box in boxes:
        show_box(box, plt.gca())
    plt.axis("on")
    plt.show()


def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels == 1]
    neg_points = coords[labels == 0]
    ax.scatter(
        pos_points[:, 0], pos_points[:, 1], color="green", marker="*",
        s=marker_size, edgecolor="white", linewidth=1.25
    )
    ax.scatter(
        neg_points[:, 0], neg_points[:, 1], color="red", marker="*",
        s=marker_size, edgecolor="white", linewidth=1.25
    )


def show_points_on_image(raw_image, input_points, input_labels=None):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    input_points = np.array(input_points)
    labels = (
        np.ones_like(input_points[:, 0])
        if input_labels is None
        else np.array(input_labels)
    )
    show_points(input_points, labels, plt.gca())
    plt.axis("on")
    plt.show()


def show_points_and_boxes_on_image(
    raw_image, boxes, input_points, input_labels=None
):
    plt.figure(figsize=(10, 10))
    plt.imshow(raw_image)
    input_points = np.array(input_points)
    labels = (
        np.ones_like(input_points[:, 0])
        if input_labels is None
        else np.array(input_labels)
    )
    show_points(input_points, labels, plt.gca())
    for box in boxes:
        show_box(box, plt.gca())
    plt.axis("on")
    plt.show()


def fig2img(fig):
    """Convert a Matplotlib figure to a PIL image and return it."""
    buf = io.BytesIO()
    fig.savefig(buf)
    buf.seek(0)
    return Image.open(buf)


def show_mask_on_image(raw_image, mask, return_image=False):
    if not isinstance(mask, torch.Tensor):
        mask = torch.tensor(mask)
    if len(mask.shape) == 4:
        mask = mask.squeeze()

    fig, axes = plt.subplots(1, 1, figsize=(15, 15))
    mask = mask.cpu().detach()
    axes.imshow(np.array(raw_image))
    show_mask(mask, axes)
    axes.axis("off")
    plt.show()

    if return_image:
        return fig2img(fig)


def show_pipe_masks_on_image(raw_image, outputs):
    plt.imshow(np.array(raw_image))
    ax = plt.gca()
    for mask in outputs["masks"]:
        show_mask(mask, ax=ax, random_color=True)
    plt.axis("off")
    plt.show()


Now we are ready to segment the images and create an image mask using the SAM model.


[Explore the Segment Anything project](https://segment-anything.com/)


## Mask Generation with SAM


The Segment Anything Model (SAM) is an image segmentation model developed by Meta AI. It can identify the precise location of either specific objects or every object in an image. SAM was released in April 2023 and is open source under the Apache 2.0 license.


SAM produces high-quality object masks from input prompts such as points or boxes and can be used to generate masks for all objects in an image. It has been trained on a dataset of 11 million images and 1.1 billion masks and demonstrates strong zero-shot performance on a variety of segmentation tasks.


SAM has three different encoder options for model instantiation: ViT-B, ViT-L, and ViT-H. These encoders have different parameter counts, with ViT-B having 91 million, ViT-L having 308 million, and ViT-H having 636 million parameters. The choice of encoder impacts the speed of inference, with ViT-H offering the best performance but at the cost of increased model size.


In July 2024, Meta AI released Segment Anything 2 (SAM 2), which is reported to be 6 times more accurate than the original SAM model at image segmentation tasks.


We will start by using the Hugging Face Transformers package to create a pipeline for mask generation with SlimSAM, a variation of the SAM model. The model and processor are loaded from the Zigeng/SlimSAM-uniform-77 repository on the Hugging Face Hub.


[View SlimSAM-uniform-77 on Hugging Face](https://huggingface.co/Zigeng/SlimSAM-uniform-77)


In [ ]:
from transformers import pipeline

model_id = "Zigeng/SlimSAM-uniform-77"
sam_pipe = pipeline("mask-generation", model_id)


Next, we will load and resize the image to (720, 375) to be compatible with the model we are using.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

image_path = Path("palestine_demo.jpg")
if not image_path.exists():
    urlretrieve(
        "https://cdn-images-1.medium.com/max/2000/1*ZJdW8tbpa7HUd3Rtuitiwg.jpeg",
        image_path,
    )


In [ ]:
from PIL import Image

raw_image = Image.open("palestine_demo.jpg")
raw_image.resize((720, 375))


*The original street image used for both automatic mask generation and point-prompt inference.*


The final step is to apply the model to the image, and then we will use show_pipe_masks_on_image to draw the masks on the image.


In [ ]:
# The higher the value of points_per_batch,
# the more efficient pipeline inference will be.
output = sam_pipe(raw_image, points_per_batch=32)
show_pipe_masks_on_image(raw_image, output)


*SlimSAM automatically generates colored masks for the objects it detects across the complete image.*


The results are outstanding, and the model managed to mask all the objects in the image. However, one drawback of this method is that inference takes too much time to be practical. A possible solution is to infer only the object selected with a point prompt.


## Faster Inference: Infer an Image and a Single Point


The next step is to load the pre-trained SlimSAM model along with its corresponding data processor.


In [ ]:
from transformers import SamModel, SamProcessor

model = SamModel.from_pretrained(model_id)
processor = SamProcessor.from_pretrained(model_id)


Let’s segment the person who is wearing the Palestinian flag. We can do this by passing any point in this region.


In [ ]:
input_points = [[[400, 500]]]


We will pass the raw image, the points we need to segment, and how we want the results returned. We will choose "pt", which refers to PyTorch, as we are using it throughout the article.


In [ ]:
inputs = processor(
    raw_image,
    input_points=input_points,
    return_tensors="pt",
)


Next, we will forward pass through the pre-trained SAM model using the processor inputs and retrieve the model’s output. The torch.no_grad() context manager disables gradient tracking because the goal is inference rather than training.


In [ ]:
import torch

with torch.no_grad():
    outputs = model(**inputs)


Now let’s predict the mask for the input image.


In [ ]:
predicted_masks = processor.image_processor.post_process_masks(
    outputs.pred_masks,
    inputs["original_sizes"],
    inputs["reshaped_input_sizes"],
)


We can see that we got one mask which refers to the object we selected as the input.


In [ ]:
len(predicted_masks)


**Expected output**

```text
1
```


We can also print the predicted mask shape.


In [ ]:
predicted_mask = predicted_masks[0]
predicted_mask.shape


**Expected output**

```text
torch.Size([1, 3, 855, 1300])
```


The pre-trained SAM model produces a single predicted mask with 3 channels and dimensions of 855 × 1300 pixels. Finally, let’s print the IoU scores.


In [ ]:
outputs.iou_scores


**Expected output**

```text
tensor([[[0.7848, 0.8657, 0.8971]]])
```


Let’s print the image with the masks. The first mask is for the selected object, and the inference time is only a fraction of the full-image mask generation we tried above.


In [ ]:
for i in range(3):
    show_mask_on_image(raw_image, predicted_mask[:, i])


*Point-prompt inference returns three candidate masks for the selected person together with their predicted IoU scores.*
